In [1]:
import sys, os
%pip install "urllib3<2"
print(sys.executable)
import numpy as np
import time
from dataclasses import dataclass
import tensorflow as tf

sys.path.insert(0, os.path.abspath("../.."))   # sandboxed/test2 → project root
from game_environments.ToyGame import ToyGame

from data_generation import *
from networks import *
from rollout import *
from updates import *
from agents import build_agents
from data import Config
from training import train

tf.random.set_seed(0)
np.random.seed(0)

cfg = Config(N=2, n_actions=2, n_states=2, ep_len=100, gamma=0.99, p_no_adv=0.4)
agents = build_agents(cfg, hidden=128)
belief_step = take_gru_step(agents)
print("TF", tf.__version__, "| Config:", cfg)

Note: you may need to restart the kernel to use updated packages.
/Users/andresho/Desktop/Research/Cooperative_MARL/venv/bin/python
TF 2.20.0 | Config: Config(N=2, n_actions=2, n_states=2, ep_len=100, gamma=0.99, p_no_adv=0.4)


In [2]:
hist = train(
    ToyGame, cfg, agents, belief_step, 
    B=2000, n_iters=1000, log_every=100,
    team_interval=10
)

iter   100 | reward(adv) 0.498 | reward(clean) 0.501 | belief acc 0.697 | 148s
iter   200 | reward(adv) 0.500 | reward(clean) 0.506 | belief acc 0.694 | 307s
iter   300 | reward(adv) 0.477 | reward(clean) 0.499 | belief acc 0.695 | 463s
iter   400 | reward(adv) 0.570 | reward(clean) 0.511 | belief acc 0.701 | 622s
iter   500 | reward(adv) 0.430 | reward(clean) 0.565 | belief acc 0.701 | 791s
iter   600 | reward(adv) 0.445 | reward(clean) 0.620 | belief acc 0.692 | 970s
iter   700 | reward(adv) 0.633 | reward(clean) 0.742 | belief acc 0.701 | 1153s
iter   800 | reward(adv) 0.662 | reward(clean) 0.912 | belief acc 0.698 | 1335s
iter   900 | reward(adv) 0.007 | reward(clean) 0.993 | belief acc 0.693 | 1529s
iter  1000 | reward(adv) 0.038 | reward(clean) 0.992 | belief acc 0.694 | 1689s


In [5]:
record = run_episodes(ToyGame, cfg, 5000, agents, belief_step)

In [6]:
print(record['reward'].shape)
total_reward = np.sum(record['reward'], axis=1)
avg_reward = np.mean(total_reward)
avg_reward2 = np.mean(record['reward'])
print(avg_reward)
print(avg_reward2)

(5000, 100)
42.6432
0.426432
